# Gassco–TTF Event Alignment

This notebook links reconstructed Norwegian gas outage announcements to the TTF trading series.

The event timestamp is based on the publication of the eligible first-revision Gassco outage message rather than the operational start of the outage. This distinction is important because an outage may begin before it is publicly reported. The objective is therefore to align each eligible Gassco announcement with the TTF market observations during which that information could have been reflected in price.

The alignment procedure applies a consistent set of rules for mapping eligible Gassco announcements to observed TTF market dates and return windows.

In [1]:
from pathlib import Path
from io import StringIO
import subprocess

import numpy as np
import pandas as pd

In [2]:
# Load the validated TTF source for event alignment

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

TTF_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "ttf_daily_validated.csv"
)

if not TTF_PATH.exists():
    raise FileNotFoundError(
        f"Validated TTF file not found: {TTF_PATH}"
    )

df_ttf = pd.read_csv(
    TTF_PATH,
    parse_dates=["trading_date"],
)

EXPECTED_TTF_COLUMNS = [
    "trading_date",
    "price",
    "open",
    "high",
    "low",
    "reported_volume",
    "reported_change",
    "simple_return",
    "log_return",
]

if df_ttf.columns.tolist() != EXPECTED_TTF_COLUMNS:
    raise ValueError(
        "Unexpected validated TTF schema.\n"
        f"Expected: {EXPECTED_TTF_COLUMNS}\n"
        f"Found:    {df_ttf.columns.tolist()}"
    )

print("Validated TTF file loaded")
print(f"Rows: {len(df_ttf):,}")
print(
    "Observed period:",
    df_ttf["trading_date"].min().date(),
    "to",
    df_ttf["trading_date"].max().date(),
)

display(df_ttf.head())

Validated TTF file loaded
Rows: 591
Observed period: 2024-07-01 to 2026-09-17


,trading_date,price,open,high,low,reported_volume,reported_change,simple_return,log_return
0,2024-07-01,33.485,34.455,35.110,33.400,68960.0,-0.0289,NaN,NaN
1,2024-07-02,33.685,33.435,33.900,33.115,69290.0,0.0060,0.005973,0.005955
2,2024-07-03,32.691,33.500,33.745,32.620,96040.0,-0.0295,-0.029509,-0.029953
3,2024-07-04,33.246,32.845,33.800,32.285,69880.0,0.0170,0.016977,0.016835
4,2024-07-05,33.067,33.555,33.975,32.875,55060.0,-0.0054,-0.005384,-0.005399


## Existing event universe

The PostgreSQL database already contains the event-reconstruction and earlier event-study tables produced during the initial SQL analysis.

Rather than recreating these transformations in Python, the existing event universe is audited before use. This preserves the separation between SQL-based event construction and Python-based statistical analysis while allowing the assumptions embedded in the database tables to be checked explicitly.

In [3]:
# Inspect the primary event table without modifying the database

schema_query = """
SELECT
    ordinal_position,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'mart'
  AND table_name = 'gassco_primary_reduction_events'
ORDER BY ordinal_position;
"""

schema_result = subprocess.run(
    ["psql", "-d", "norway_ttf", "-P", "pager=off",
     "-c", schema_query],
    capture_output=True,
    text=True
)

print(schema_result.stdout)

 ordinal_position |       column_name        |          data_type          
------------------+--------------------------+-----------------------------
                1 | event_key                | text
                2 | message_id               | text
                3 | asset                    | text
                4 | event_type               | text
                5 | unavailability_type      | text
                6 | event_status             | text
                7 | publication_time         | timestamp without time zone
                8 | event_start              | timestamp without time zone
                9 | event_stop               | timestamp without time zone
               10 | lead_hours               | numeric
               11 | duration_hours           | numeric
               12 | technical_capacity       | numeric
               13 | available_capacity       | numeric
               14 | unavailable_capacity     | numeric
               15 | outage_size_msm3

## Event-universe validation

The primary event table contains 166 reconstructed reduction events. Before market observations are attached, the event universe is checked for identifier uniqueness, missing analytical fields, revision composition and publication timing.

This ensures that the subsequent event study operates on the intended event population rather than relying only on the table name or earlier processing assumptions.

In [4]:
# Inspect revision and publication-timing composition

composition_query = """
SELECT
    revision,
    publication_timing_group,
    COUNT(*) AS n_events
FROM mart.gassco_primary_reduction_events
GROUP BY revision, publication_timing_group
ORDER BY revision, publication_timing_group;
"""

composition_result = subprocess.run(
    [
        "psql", "-d", "norway_ttf",
        "-P", "pager=off",
        "-c", composition_query
    ],
    capture_output=True,
    text=True
)

print(composition_result.stdout)

 revision |   publication_timing_group   | n_events 
----------+------------------------------+----------
        1 | published_after_start        |      110
        1 | published_before_or_at_start |       56
(2 rows)




In [5]:
# Audit the primary Gassco event universe

audit_query = """
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT event_key) AS unique_event_keys,
    COUNT(DISTINCT message_id) AS unique_message_ids,

    COUNT(*) FILTER (
        WHERE publication_time IS NULL
    ) AS missing_publication_time,

    COUNT(*) FILTER (
        WHERE event_start IS NULL
    ) AS missing_event_start,
    
    COUNT(*) FILTER (
    WHERE outage_size_msm3d IS NULL
) AS missing_outage_size,

    MIN(publication_time) AS first_publication,
    MAX(publication_time) AS last_publication

FROM mart.gassco_primary_reduction_events;
"""

audit_result = subprocess.run(
    [
        "psql",
        "-d", "norway_ttf",
        "-P", "pager=off",
        "-c", audit_query
    ],
    capture_output=True,
    text=True
)

print("AUDIT RESULTS")
print("-------------")
print(audit_result.stdout)

if audit_result.stderr:
    print("ERROR:")
    print(audit_result.stderr)

print("Return code:", audit_result.returncode)

AUDIT RESULTS
-------------
 rows | unique_event_keys | unique_message_ids | missing_publication_time | missing_event_start | missing_outage_size |  first_publication  |  last_publication   
------+-------------------+--------------------+--------------------------+---------------------+---------------------+---------------------+---------------------
  166 |               166 |                166 |                        0 |                   0 |                   0 | 2024-09-03 13:10:34 | 2026-08-31 05:53:54
(1 row)


Return code: 0


### Event-universe findings

The primary event universe contains 166 observations representing 166 unique event keys and 166 unique Gassco message IDs. All events contain both a publication timestamp and an operational start timestamp.

Eligible event publications span 3 September 2024 to 31 August 2026. The available TTF series extends beyond the event sample at both ends, providing market observations before the first eligible event and after the final eligible event.

All primary events are first-revision messages. Of the 166 events, 110 were published after the reported operational start of the outage and 56 were published before or at the operational start. Publication time is therefore retained as the event timestamp used for market alignment.

In [6]:


event_query = """
SELECT *
FROM mart.gassco_primary_reduction_events
ORDER BY publication_time;
"""

event_result = subprocess.run(
    [
        "psql",
        "-d", "norway_ttf",
        "--csv",
        "-c", event_query
    ],
    capture_output=True,
    text=True,
    check=True
)

df_events = pd.read_csv(
    StringIO(event_result.stdout),
    parse_dates=[
        "publication_time",
        "event_start",
        "event_stop"
    ]
)

print("Shape:", df_events.shape)
print("\nColumns:")
print(df_events.columns.tolist())

display(df_events.head())

Shape: (166, 22)

Columns:
['event_key', 'message_id', 'asset', 'event_type', 'unavailability_type', 'event_status', 'publication_time', 'event_start', 'event_stop', 'lead_hours', 'duration_hours', 'technical_capacity', 'available_capacity', 'unavailable_capacity', 'outage_size_msm3d', 'reason', 'remarks', 'source_year', 'source_file', 'source_excel_row', 'revision', 'publication_timing_group']


,event_key,message_id,asset,event_type,unavailability_type,event_status,publication_time,event_start,event_stop,lead_hours,...,available_capacity,unavailable_capacity,outage_size_msm3d,reason,remarks,source_year,source_file,source_excel_row,revision,publication_timing_group
0,2024-342,2024-342__________________001,Kollsnes,Other unavailability,Unplanned,Inactive,2024-09-03 13:10:34,2024-09-03 06:00:00,2024-09-04 06:00:00,-7.176111,...,102.0,14.0,14.0,Downstream restriction,NaN,2024,gassco_umm_unplanned_2024.xlsx,94,1,published_after_start
1,2024-346,2024-346__________________001,Gullfaks,Other unavailability,Unplanned,Inactive,2024-09-04 22:13:23,2024-09-04 06:00:00,2024-09-05 06:00:00,-16.223056,...,0.0,15.0,15.0,Compressor failure,NaN,2024,gassco_umm_unplanned_2024.xlsx,92,1,published_after_start
2,2024-348,2024-348__________________001,Gullfaks,Other unavailability,Unplanned,Inactive,2024-09-07 03:34:57,2024-09-07 06:00:00,2024-09-08 06:00:00,2.417500,...,4.5,10.0,10.0,Process problems,Uncertain duration,2024,gassco_umm_unplanned_2024.xlsx,91,1,published_before_or_at_start
3,2024-352,2024-352__________________001,Dvalin,Gas production field unavailability,Unplanned,Inactive,2024-09-08 20:04:54,2024-09-09 06:00:00,2024-09-10 06:00:00,9.918333,...,2.0,7.0,7.0,Process problems,Uncertain duration,2024,gassco_umm_unplanned_2024.xlsx,89,1,published_before_or_at_start
4,2024-359,2024-359__________________001,Troll,Other unavailability,Unplanned,Inactive,2024-09-14 06:57:00,2024-09-14 06:00:00,2024-09-16 06:00:00,-0.950000,...,93.4,37.6,37.6,Corrective maintenance,shadow maintenance,2024,gassco_umm_unplanned_2024.xlsx,88,1,published_after_start


## Event-time alignment

The event study is designed around the publication of eligible first-revision Gassco outage messages rather than the physical start of an outage. `publication_time` is therefore used as the event timestamp. `event_start` is retained separately to describe whether the market was informed before or after the reported operational disruption began.

The market series is the Investing.com ICE Dutch TTF Gas C1 Futures historical series. It provides one observation per trading date rather than intraday prices. 

This creates an important identification constraint. A Gassco announcement may occur during a trading day, after trading, or on a non-trading day, but the daily market series does not reveal the price immediately before and immediately after the announcement. The analysis therefore measures the response across observable daily trading prices rather than claiming an instantaneous intraday reaction.

Event alignment follows the observed TTF trading calendar rather than ordinary calendar days. Announcements made on dates without an observed TTF price are carried forward to the next available TTF observation.

For announcements occurring on a TTF trading date, the publication date is retained as the calendar anchor for the daily market window. Subsequent return windows are constructed from the ordered sequence of observed TTF trading dates. 

This distinction prevents calendar gaps and publication timing from being treated as if they represented identical market-information windows.

In [7]:
# Classify Gassco announcements against observed TTF trading dates

event_alignment = df_events.copy()

# Calendar date on which the Gassco message was published
event_alignment["publication_date"] = (
    event_alignment["publication_time"].dt.normalize()
)

# Observed dates in the validated TTF series
ttf_dates = pd.Index(
    df_ttf["trading_date"].dt.normalize().unique()
)

# Does a TTF observation exist on the publication date?
event_alignment["publication_date_is_ttf_date"] = (
    event_alignment["publication_date"].isin(ttf_dates)
)

# Useful diagnostic label
event_alignment["publication_weekday"] = (
    event_alignment["publication_time"].dt.day_name()
)

print("Total events:", len(event_alignment))

print("\nPublication date represented in TTF series:")
print(
    event_alignment["publication_date_is_ttf_date"]
    .value_counts()
)

print("\nPublication weekday:")
print(
    event_alignment["publication_weekday"]
    .value_counts()
)

print("\nEvents published without a same-date TTF observation:")

display(
    event_alignment.loc[
        ~event_alignment["publication_date_is_ttf_date"],
        [
            "event_key",
            "asset",
            "publication_time",
            "publication_weekday",
            "event_start"
        ]
    ]
    .sort_values("publication_time")
    .reset_index(drop=True)
)

Total events: 166

Publication date represented in TTF series:
publication_date_is_ttf_date
True     117
False     49
Name: count, dtype: int64

Publication weekday:
publication_weekday
Sunday       30
Monday       28
Saturday     24
Wednesday    23
Thursday     21
Friday       21
Tuesday      19
Name: count, dtype: int64

Events published without a same-date TTF observation:


,event_key,asset,publication_time,publication_weekday,event_start
0,2024-348,Gullfaks,2024-09-07 03:34:57,Saturday,2024-09-07 06:00:00
1,2024-352,Dvalin,2024-09-08 20:04:54,Sunday,2024-09-09 06:00:00
2,2024-359,Troll,2024-09-14 06:57:00,Saturday,2024-09-14 06:00:00
3,2024-373,Kårstø,2024-09-21 09:11:30,Saturday,2024-09-21 09:10:00
4,2024-374,Dvalin,2024-09-21 10:42:16,Saturday,2024-09-21 10:40:00
5,2024-376,Kårstø,2024-09-22 08:42:42,Sunday,2024-09-22 06:00:00
6,2024-379,Troll,2024-09-22 20:23:44,Sunday,2024-09-22 06:00:00
7,2024-381-A,Åsgard,2024-09-29 07:41:40,Sunday,2024-09-29 06:00:00
8,2024-415,Troll,2024-10-19 10:11:59,Saturday,2024-10-19 06:00:00
9,2024-422-A,Kollsnes,2024-10-27 20:07:24,Sunday,2024-10-28 06:00:00


### Calendar-alignment findings

Of the 166 eligible Gassco outage announcements, 117 were published on dates represented in the observed TTF series, while 49 were published on dates without a same-date TTF observation.

The 49 unmatched publication dates shown in the diagnostic are weekend publications. These events do not have a same-date TTF price observation and are therefore mapped forward to the next observed TTF date.

This resolves the calendar component of event alignment. It does not resolve the separate intraday timing problem for the 117 announcements published on dates with a TTF observation.  For those events, the publication timestamp must still be considered when assessing how much of the same-date TTF price movement could plausibly occur after publication.

In [8]:
# Map each publication to the first observable TTF trading date
# on or after its publication date

ttf_calendar = (
    df_ttf[["trading_date"]]
    .drop_duplicates()
    .sort_values("trading_date")
    .reset_index(drop=True)
)

event_alignment = (
    event_alignment
    .sort_values("publication_date")
    .reset_index(drop=True)
)

event_alignment = pd.merge_asof(
    event_alignment,
    ttf_calendar.rename(
        columns={"trading_date": "first_available_ttf_date"}
    ),
    left_on="publication_date",
    right_on="first_available_ttf_date",
    direction="forward"
)

# Calendar delay between publication date and first available TTF date
event_alignment["calendar_delay_days"] = (
    event_alignment["first_available_ttf_date"]
    - event_alignment["publication_date"]
).dt.days

print("Events:", len(event_alignment))

print("\nCalendar delay to first available TTF observation:")
print(
    event_alignment["calendar_delay_days"]
    .value_counts()
    .sort_index()
)

print("\nMaximum calendar delay:")
print(event_alignment["calendar_delay_days"].max())

print("\nEvents requiring forward mapping:")
display(
    event_alignment.loc[
        event_alignment["calendar_delay_days"] > 0,
        [
            "event_key",
            "asset",
            "publication_time",
            "publication_weekday",
            "first_available_ttf_date",
            "calendar_delay_days"
        ]
    ]
    .sort_values("publication_time")
    .reset_index(drop=True)
)

Events: 166

Calendar delay to first available TTF observation:
calendar_delay_days
0    117
1     30
2     18
3      1
Name: count, dtype: int64

Maximum calendar delay:
3

Events requiring forward mapping:


,event_key,asset,publication_time,publication_weekday,first_available_ttf_date,calendar_delay_days
0,2024-348,Gullfaks,2024-09-07 03:34:57,Saturday,2024-09-09,2
1,2024-352,Dvalin,2024-09-08 20:04:54,Sunday,2024-09-09,1
2,2024-359,Troll,2024-09-14 06:57:00,Saturday,2024-09-16,2
3,2024-373,Kårstø,2024-09-21 09:11:30,Saturday,2024-09-23,2
4,2024-374,Dvalin,2024-09-21 10:42:16,Saturday,2024-09-23,2
5,2024-376,Kårstø,2024-09-22 08:42:42,Sunday,2024-09-23,1
6,2024-379,Troll,2024-09-22 20:23:44,Sunday,2024-09-23,1
7,2024-381-A,Åsgard,2024-09-29 07:41:40,Sunday,2024-09-30,1
8,2024-415,Troll,2024-10-19 10:11:59,Saturday,2024-10-21,2
9,2024-422-A,Kollsnes,2024-10-27 20:07:24,Sunday,2024-10-28,1


### Forward-mapping results

The observed TTF trading calendar provides a same-date market observation for 117 of the 166 eligible Gassco announcements.

The remaining 49 events require forward alignment to the first subsequent date observed in the TTF series. Of these, 30 are mapped forward by one calendar day, 18 by two calendar days and one by three calendar days. The maximum calendar delay is therefore three days.

Forward mapping is based on the observed TTF series rather than an assumed Monday-to-Friday calendar. This allows weekends and longer market-data gaps to be handled consistently without creating artificial price observations.

The resulting `first_available_ttf_date` represents the earliest observed TTF date on or after publication. It is not yet treated as the final event date because announcements made on an existing TTF trading date still differ materially in their intraday publication timing.

## Intraday publication timing

For announcements published on dates with an observed TTF price, calendar alignment alone is insufficient.

An announcement released early in the trading day provides substantially more time for new information to be reflected in the same-date TTF price observation than an announcement released later in the day. Treating both cases identically could therefore misclassify the market-response window.

Before specifying the final event-day convention, the distribution of publication times is examined for the 117 announcements with a same-date TTF observation.

In [9]:
# Examine publication timing for events with a same-date TTF observation

same_day_events = event_alignment.loc[
    event_alignment["calendar_delay_days"] == 0
].copy()

same_day_events["publication_hour"] = (
    same_day_events["publication_time"].dt.hour
    + same_day_events["publication_time"].dt.minute / 60
    + same_day_events["publication_time"].dt.second / 3600
)

print("Same-date events:", len(same_day_events))

print("\nPublication-hour summary:")
print(
    same_day_events["publication_hour"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

print("\nPublication counts by clock hour:")
print(
    same_day_events["publication_time"]
    .dt.hour
    .value_counts()
    .sort_index()
)

print("\nEarliest same-date publications:")
display(
    same_day_events[
        [
            "event_key",
            "asset",
            "publication_time",
            "first_available_ttf_date"
        ]
    ]
    .sort_values("publication_time", key=lambda x: x.dt.time)
    .head(10)
)

print("\nLatest same-date publications:")
display(
    same_day_events[
        [
            "event_key",
            "asset",
            "publication_time",
            "first_available_ttf_date"
        ]
    ]
    .sort_values(
        "publication_time",
        key=lambda x: x.dt.time,
        ascending=False
    )
    .head(10)
)

Same-date events: 117

Publication-hour summary:
count    117.000000
mean      13.246180
std        5.745195
min        0.009444
10%        6.602333
25%        9.946389
50%       13.176111
75%       17.561667
90%       20.929278
max       23.553333
Name: publication_hour, dtype: float64

Publication counts by clock hour:
publication_time
0      4
1      2
2      2
3      2
5      2
7      7
8      6
9      5
10     8
11    11
12     9
13     8
14     6
15     9
16     5
17     3
18     4
19     8
20     4
21     4
22     6
23     2
Name: count, dtype: int64

Earliest same-date publications:


,event_key,asset,publication_time,first_available_ttf_date
20,2024-401-B,Oseberg,2024-10-14 00:00:34,2024-10-14
19,2024-393-C,Kristin,2024-10-14 00:15:33,2024-10-14
28,2024-430,Visund,2024-11-05 00:25:30,2024-11-05
57,2025-203,Åsgard,2025-03-21 00:40:15,2025-03-21
129,2026-203,Troll,2026-05-07 01:06:00,2026-05-07
98,2025-378,Norne,2025-10-22 01:12:12,2025-10-22
112,2026-127,Kristin,2026-01-21 02:04:25,2026-01-21
11,2024-345-A,Gullfaks,2024-09-23 02:20:46,2024-09-23
15,2024-386-A,Sleipner,2024-09-26 03:02:43,2024-09-26
49,2025-174,Oseberg,2025-02-11 03:33:33,2025-02-11



Latest same-date publications:


,event_key,asset,publication_time,first_available_ttf_date
73,2025-247,Aasta Hansteen,2025-05-23 23:33:12,2025-05-23
26,2024-104-A,Johan Sverdrup,2024-11-01 23:31:47,2024-11-01
80,2025-287,Nyhamna,2025-07-15 22:51:31,2025-07-15
74,2025-256,Fields Delivering into SEGAL,2025-06-05 22:36:31,2025-06-05
143,2026-244-A,Kollsnes,2026-06-15 22:28:08,2026-06-15
127,2026-200-A,Åsgard,2026-05-03 22:20:49,2026-05-03
145,2026-15-A,Kollsnes,2026-06-17 22:14:34,2026-06-17
1,2024-346,Gullfaks,2024-09-04 22:13:23,2024-09-04
90,2025-345-A,Skarv,2025-09-22 21:47:51,2025-09-22
56,2025-202,Troll,2025-03-20 21:46:55,2025-03-20


### Intraday timing findings

Publication timing is material to the event-study design. Among the 117 announcements published on dates with a TTF observation, publication times are distributed throughout the day rather than concentrated in the morning.

The median publication time is approximately 13:11, the 75th percentile is approximately 17:34 and the 90th percentile is approximately 20:56. The sample also contains announcements published late in the evening, including after 22:00.

A simple calendar-date merge would therefore risk treating the same-date TTF price observation as post-publication even when the announcement occurred late in the day.

Before an intraday alignment rule is imposed, the timezone convention of the Gassco publication timestamps must be established. The database currently stores `publication_time` as a timestamp without timezone information. Comparing these clock times directly with ICE trading hours without establishing their timezone would introduce an unsupported timing assumption.

No precise intraday cut-off is imposed in the primary alignment design.

### Publication-time timezone

Gassco's publication principles identify `Time of Publication` as a reported field for outage events and state that information on unplanned events is published as soon as practically possible. This supports the use of publication time, rather than operational outage start, as the event timestamp for the first-revision message.

However, the available Gassco documentation reviewed for this analysis does not explicitly specify the timezone attached to the historical `Time of Publication` field. The database also stores `publication_time` as a timestamp without timezone information.

The publication timestamps are therefore not assigned an assumed UTC, CET or CEST timezone. This prevents an unsupported conversion from being introduced into the event-study design.

As a consequence, the analysis does not use a precise exchange-closing-time cut-off as its primary alignment rule. Intraday publication timing is instead treated as a source of timing uncertainty and addressed through alternative event-alignment specifications.

## Event-window specification

Because the market dataset contains one daily TTF price observation rather than intraday prices, no single daily return can perfectly isolate the immediate response to an announcement released during the trading day.

For announcements published on a date with a TTF observation, the publication-day return may contain price movement that occurred before the announcement. Conversely, using only the return to the next observed TTF date may omit information already reflected in the publication-date price.

The analysis therefore avoids relying on a single intraday cut-off. Market responses are evaluated across alternative daily windows that distinguish the publication-day return, the return to the next observed TTF price and a wider window spanning both observations.

For announcements published on dates without a TTF observation, alignment begins with the first subsequent observed TTF trading date.

This approach treats the mismatch between timestamped outage information and daily market prices as an identification limitation rather than introducing unsupported intraday precision. Results will be compared across the alternative windows as a robustness test.

## Construct event trading windows

Each TTF observation is assigned a sequential trading index. Gassco events are then mapped onto this observed trading sequence rather than calendar-day offsets.

For events published on a date with a TTF observation, the publication-date observation is retained as the daily market anchor. For events published without a same-date observation, the first subsequent TTF observation is used as the first observable post-publication market date.

The surrounding trading observations are then identified mechanically from the ordered TTF trading sequence. This produces the dates required for the alternative event-return definitions.

In [10]:
# Construct event trading-window indices

# -----------------------------------
# 1. Create sequential TTF trading index
# -----------------------------------

ttf_indexed = (
    df_ttf
    .sort_values("trading_date")
    .reset_index(drop=True)
    .copy()
)

ttf_indexed["ttf_index"] = np.arange(len(ttf_indexed))

# Lookup: trading date -> sequential index
date_to_index = (
    ttf_indexed
    .set_index("trading_date")["ttf_index"]
)


# -----------------------------------
# 2. Identify first observable TTF date
# -----------------------------------

event_windows = event_alignment.copy()

event_windows["anchor_index"] = (
    event_windows["first_available_ttf_date"]
    .map(date_to_index)
)

if event_windows["anchor_index"].isna().any():
    raise ValueError(
        "At least one event could not be mapped to the TTF trading calendar."
    )

event_windows["anchor_index"] = (
    event_windows["anchor_index"].astype(int)
)


# -----------------------------------
# 3. Create surrounding trading indices
# -----------------------------------

event_windows["index_m1"] = event_windows["anchor_index"] - 1
event_windows["index_0"]  = event_windows["anchor_index"]
event_windows["index_p1"] = event_windows["anchor_index"] + 1
event_windows["index_p2"] = event_windows["anchor_index"] + 2


# -----------------------------------
# 4. Map indices back to trading dates
# -----------------------------------

index_to_date = (
    ttf_indexed
    .set_index("ttf_index")["trading_date"]
)

event_windows["date_m1"] = (
    event_windows["index_m1"].map(index_to_date)
)

event_windows["date_0"] = (
    event_windows["index_0"].map(index_to_date)
)

event_windows["date_p1"] = (
    event_windows["index_p1"].map(index_to_date)
)

event_windows["date_p2"] = (
    event_windows["index_p2"].map(index_to_date)
)


# -----------------------------------
# 5. Validate the mapping
# -----------------------------------

print("Events:", len(event_windows))

print(
    "Missing anchor dates:",
    event_windows["date_0"].isna().sum()
)

print(
    "Missing previous trading dates:",
    event_windows["date_m1"].isna().sum()
)

print(
    "Missing +1 trading dates:",
    event_windows["date_p1"].isna().sum()
)

print(
    "Missing +2 trading dates:",
    event_windows["date_p2"].isna().sum()
)

print("\nAnchor-date reconciliation:")
print(
    (
        event_windows["date_0"]
        == event_windows["first_available_ttf_date"]
    ).value_counts()
)

display(
    event_windows[
        [
            "event_key",
            "asset",
            "publication_time",
            "publication_date",
            "calendar_delay_days",
            "date_m1",
            "date_0",
            "date_p1",
            "date_p2"
        ]
    ].head(10)
)

Events: 166
Missing anchor dates: 0
Missing previous trading dates: 0
Missing +1 trading dates: 0
Missing +2 trading dates: 0

Anchor-date reconciliation:
True    166
Name: count, dtype: int64


,event_key,asset,publication_time,publication_date,calendar_delay_days,date_m1,date_0,date_p1,date_p2
0,2024-342,Kollsnes,2024-09-03 13:10:34,2024-09-03,0,2024-09-02,2024-09-03,2024-09-04,2024-09-05
1,2024-346,Gullfaks,2024-09-04 22:13:23,2024-09-04,0,2024-09-03,2024-09-04,2024-09-05,2024-09-06
2,2024-348,Gullfaks,2024-09-07 03:34:57,2024-09-07,2,2024-09-06,2024-09-09,2024-09-10,2024-09-11
3,2024-352,Dvalin,2024-09-08 20:04:54,2024-09-08,1,2024-09-06,2024-09-09,2024-09-10,2024-09-11
4,2024-359,Troll,2024-09-14 06:57:00,2024-09-14,2,2024-09-13,2024-09-16,2024-09-17,2024-09-18
5,2024-362,Oseberg,2024-09-16 09:14:25,2024-09-16,0,2024-09-13,2024-09-16,2024-09-17,2024-09-18
6,2024-372,Heidrun,2024-09-19 19:57:03,2024-09-19,0,2024-09-18,2024-09-19,2024-09-20,2024-09-23
7,2024-373,Kårstø,2024-09-21 09:11:30,2024-09-21,2,2024-09-20,2024-09-23,2024-09-24,2024-09-25
8,2024-374,Dvalin,2024-09-21 10:42:16,2024-09-21,2,2024-09-20,2024-09-23,2024-09-24,2024-09-25
9,2024-379,Troll,2024-09-22 20:23:44,2024-09-22,1,2024-09-20,2024-09-23,2024-09-24,2024-09-25


## Construct market-response windows

The validated event anchors are now linked to TTF prices at the surrounding observed trading dates.

Three complementary return windows are constructed:

- **Publication/anchor return:** the log return from the previous observed TTF price to the anchor-date price, measured from \(t-1\) to \(t=0\).
- **Subsequent return:** the log return from the anchor-date price to the next observed TTF price, measured from \(t=0\) to \(t+1\).
- **Two-observation response:** the cumulative log return from \(t-1\) to \(t+1\).

For same-date events, the first measure may contain market movement that occurred before publication, while the subsequent measure may omit information already reflected in the publication-date price. The wider two-observation window is therefore retained as a complementary specification.

For events published on dates without a TTF observation, \(t=0\) is the first subsequent observed TTF date.

These windows are applied consistently across all aligned events.

In [11]:
# Attach TTF prices to event-window dates and calculate returns

# -----------------------------------
# 1. Create price lookup
# -----------------------------------

price_lookup = (
    ttf_indexed
    .set_index("ttf_index")["price"]
)

# -----------------------------------
# 2. Attach prices
# -----------------------------------

event_windows["price_m1"] = (
    event_windows["index_m1"].map(price_lookup)
)

event_windows["price_0"] = (
    event_windows["index_0"].map(price_lookup)
)

event_windows["price_p1"] = (
    event_windows["index_p1"].map(price_lookup)
)

event_windows["price_p2"] = (
    event_windows["index_p2"].map(price_lookup)
)

# -----------------------------------
# 3. Calculate log-return windows
# -----------------------------------

# Close immediately before anchor -> anchor close
event_windows["return_m1_0"] = np.log(
    event_windows["price_0"]
    / event_windows["price_m1"]
)

# Anchor close -> next observed close
event_windows["return_0_p1"] = np.log(
    event_windows["price_p1"]
    / event_windows["price_0"]
)

# Previous close -> next observed close
event_windows["return_m1_p1"] = np.log(
    event_windows["price_p1"]
    / event_windows["price_m1"]
)

# Anchor close -> second subsequent close
event_windows["return_0_p2"] = np.log(
    event_windows["price_p2"]
    / event_windows["price_0"]
)

# -----------------------------------
# 4. Validate price attachment
# -----------------------------------

price_columns = [
    "price_m1",
    "price_0",
    "price_p1",
    "price_p2"
]

return_columns = [
    "return_m1_0",
    "return_0_p1",
    "return_m1_p1",
    "return_0_p2"
]

print("Events:", len(event_windows))

print("\nMissing attached prices:")
print(
    event_windows[price_columns]
    .isna()
    .sum()
)

print("\nMissing calculated returns:")
print(
    event_windows[return_columns]
    .isna()
    .sum()
)

print("\nNon-finite calculated returns:")
print(
    (~np.isfinite(event_windows[return_columns]))
    .sum()
)

# Algebraic check:
# log(P0/P-1) + log(P+1/P0)
# must equal log(P+1/P-1)

event_windows["return_identity_error"] = (
    event_windows["return_m1_0"]
    + event_windows["return_0_p1"]
    - event_windows["return_m1_p1"]
)

print(
    "\nMaximum absolute return identity error:",
    event_windows["return_identity_error"]
    .abs()
    .max()
)

# -----------------------------------
# 5. Inspect mapping only
# -----------------------------------

display(
    event_windows[
        [
            "event_key",
            "publication_time",
            "calendar_delay_days",
            "date_m1",
            "price_m1",
            "date_0",
            "price_0",
            "date_p1",
            "price_p1",
            "date_p2",
            "price_p2"
        ]
    ].head(10)
)

Events: 166

Missing attached prices:
price_m1    0
price_0     0
price_p1    0
price_p2    0
dtype: int64

Missing calculated returns:
return_m1_0     0
return_0_p1     0
return_m1_p1    0
return_0_p2     0
dtype: int64

Non-finite calculated returns:
return_m1_0     0
return_0_p1     0
return_m1_p1    0
return_0_p2     0
dtype: int64

Maximum absolute return identity error: 2.220446049250313e-16


,event_key,publication_time,calendar_delay_days,date_m1,price_m1,date_0,price_0,date_p1,price_p1,date_p2,price_p2
0,2024-342,2024-09-03 13:10:34,0,2024-09-02,38.580,2024-09-03,37.193,2024-09-04,35.798,2024-09-05,36.195
1,2024-346,2024-09-04 22:13:23,0,2024-09-03,37.193,2024-09-04,35.798,2024-09-05,36.195,2024-09-06,36.479
2,2024-348,2024-09-07 03:34:57,2,2024-09-06,36.479,2024-09-09,37.333,2024-09-10,35.282,2024-09-11,36.142
3,2024-352,2024-09-08 20:04:54,1,2024-09-06,36.479,2024-09-09,37.333,2024-09-10,35.282,2024-09-11,36.142
4,2024-359,2024-09-14 06:57:00,2,2024-09-13,35.648,2024-09-16,34.081,2024-09-17,35.547,2024-09-18,35.245
5,2024-362,2024-09-16 09:14:25,0,2024-09-13,35.648,2024-09-16,34.081,2024-09-17,35.547,2024-09-18,35.245
6,2024-372,2024-09-19 19:57:03,0,2024-09-18,35.245,2024-09-19,33.083,2024-09-20,34.444,2024-09-23,36.211
7,2024-373,2024-09-21 09:11:30,2,2024-09-20,34.444,2024-09-23,36.211,2024-09-24,35.413,2024-09-25,37.319
8,2024-374,2024-09-21 10:42:16,2,2024-09-20,34.444,2024-09-23,36.211,2024-09-24,35.413,2024-09-25,37.319
9,2024-379,2024-09-22 20:23:44,1,2024-09-20,34.444,2024-09-23,36.211,2024-09-24,35.413,2024-09-25,37.319


## Event clustering and overlapping market windows

Multiple Gassco announcements can occur on the same date or sufficiently close together that they map to the same TTF trading observations.

This creates an important dependence problem. If several announcements share the same market-response window, treating them as independent observations would repeatedly count the same TTF price movement. Closely spaced announcements may also make it difficult to attribute a subsequent market movement to one individual outage message.

The degree of event clustering is therefore quantified at both the anchor-date and response-window level. The analysis therefore distinguishes individual outage announcements from unique market-response windows, so announcements sharing the same TTF anchor are not counted as distinct market observations.

In [12]:
# Diagnose event clustering and overlapping TTF windows

# -----------------------------------
# 1. Events sharing the same anchor date
# -----------------------------------

anchor_clusters = (
    event_windows
    .groupby("date_0")
    .agg(
        n_events=("event_key", "size"),
        n_assets=("asset", "nunique")
    )
    .reset_index()
    .sort_values(
        ["n_events", "date_0"],
        ascending=[False, True]
    )
)

print("Total Gassco events:", len(event_windows))
print(
    "Unique TTF anchor dates:",
    event_windows["date_0"].nunique()
)

print(
    "Anchor dates containing multiple events:",
    (anchor_clusters["n_events"] > 1).sum()
)

print(
    "Events occurring on multi-event anchor dates:",
    anchor_clusters.loc[
        anchor_clusters["n_events"] > 1,
        "n_events"
    ].sum()
)

print("\nEvents per anchor date:")
print(
    anchor_clusters["n_events"]
    .value_counts()
    .sort_index()
)


# -----------------------------------
# 2. Exact duplicate response windows
# -----------------------------------

window_clusters = (
    event_windows
    .groupby(
        ["date_m1", "date_0", "date_p1", "date_p2"]
    )
    .agg(
        n_events=("event_key", "size"),
        n_assets=("asset", "nunique")
    )
    .reset_index()
    .sort_values(
        ["n_events", "date_0"],
        ascending=[False, True]
    )
)

print(
    "\nUnique exact TTF windows:",
    len(window_clusters)
)

print(
    "Exact windows containing multiple events:",
    (window_clusters["n_events"] > 1).sum()
)

print(
    "Events contained in duplicated exact windows:",
    window_clusters.loc[
        window_clusters["n_events"] > 1,
        "n_events"
    ].sum()
)


# -----------------------------------
# 3. Show clustered events
# -----------------------------------

clustered_anchor_dates = set(
    anchor_clusters.loc[
        anchor_clusters["n_events"] > 1,
        "date_0"
    ]
)

clustered_events = (
    event_windows.loc[
        event_windows["date_0"].isin(clustered_anchor_dates),
        [
            "event_key",
            "asset",
            "publication_time",
            "calendar_delay_days",
            "date_m1",
            "date_0",
            "date_p1",
            "date_p2"
        ]
    ]
    .sort_values(
        ["date_0", "publication_time"]
    )
    .reset_index(drop=True)
)

print("\nEvents sharing a TTF anchor date:")
display(clustered_events)

Total Gassco events: 166
Unique TTF anchor dates: 130
Anchor dates containing multiple events: 22
Events occurring on multi-event anchor dates: 58

Events per anchor date:
n_events
1    108
2     13
3      6
4      2
6      1
Name: count, dtype: int64

Unique exact TTF windows: 130
Exact windows containing multiple events: 22
Events contained in duplicated exact windows: 58

Events sharing a TTF anchor date:


,event_key,asset,publication_time,calendar_delay_days,date_m1,date_0,date_p1,date_p2
0,2024-348,Gullfaks,2024-09-07 03:34:57,2,2024-09-06,2024-09-09,2024-09-10,2024-09-11
1,2024-352,Dvalin,2024-09-08 20:04:54,1,2024-09-06,2024-09-09,2024-09-10,2024-09-11
2,2024-359,Troll,2024-09-14 06:57:00,2,2024-09-13,2024-09-16,2024-09-17,2024-09-18
3,2024-362,Oseberg,2024-09-16 09:14:25,0,2024-09-13,2024-09-16,2024-09-17,2024-09-18
4,2024-373,Kårstø,2024-09-21 09:11:30,2,2024-09-20,2024-09-23,2024-09-24,2024-09-25
5,2024-374,Dvalin,2024-09-21 10:42:16,2,2024-09-20,2024-09-23,2024-09-24,2024-09-25
6,2024-376,Kårstø,2024-09-22 08:42:42,1,2024-09-20,2024-09-23,2024-09-24,2024-09-25
7,2024-379,Troll,2024-09-22 20:23:44,1,2024-09-20,2024-09-23,2024-09-24,2024-09-25
8,2024-345-A,Gullfaks,2024-09-23 02:20:46,0,2024-09-20,2024-09-23,2024-09-24,2024-09-25
9,2024-380,Sleipner,2024-09-23 10:30:35,0,2024-09-20,2024-09-23,2024-09-24,2024-09-25


### Event-clustering findings

The 166 eligible Gassco announcements map to 130 unique TTF anchor dates. Most anchor dates contain a single announcement, but clustering is material: 22 anchor dates contain multiple events, accounting for 58 of the 166 announcements.

Among the 130 unique anchor dates, 108 contain one event, 13 contain two, six contain three, two contain four and one contains six announcements.

Because the surrounding TTF observations are determined by the anchor date, announcements sharing an anchor also share the same market-response window.Treating all 166 announcements as separate market observations would therefore duplicate some TTF returns and overstate the number of distinct market-response windows.

The analysis consequently retains the announcement-level dataset for outage characteristics and descriptive analysis, while recognising the unique TTF anchor date as the relevant unit when analysing distinct market-return observations.

In [13]:
# Diagnose overlap between unique TTF event anchors

unique_anchors = (
    event_windows[
        [
            "date_0",
            "anchor_index"
        ]
    ]
    .drop_duplicates()
    .sort_values("anchor_index")
    .reset_index(drop=True)
)

# Distance from previous event anchor in TTF trading observations
unique_anchors["previous_anchor_index"] = (
    unique_anchors["anchor_index"].shift(1)
)

unique_anchors["previous_anchor_date"] = (
    unique_anchors["date_0"].shift(1)
)

unique_anchors["trading_gap"] = (
    unique_anchors["anchor_index"]
    - unique_anchors["previous_anchor_index"]
)

print("Unique TTF anchor dates:", len(unique_anchors))

print("\nTrading-observation gap from previous anchor:")
print(
    unique_anchors["trading_gap"]
    .value_counts()
    .sort_index()
)

print(
    "\nAnchors immediately following another event anchor:",
    (unique_anchors["trading_gap"] == 1).sum()
)

print(
    "Anchors within two trading observations of previous anchor:",
    (unique_anchors["trading_gap"] <= 2).sum()
)

print(
    "Anchors within three trading observations of previous anchor:",
    (unique_anchors["trading_gap"] <= 3).sum()
)

print("\nClosest consecutive event anchors:")

display(
    unique_anchors.loc[
        unique_anchors["trading_gap"] <= 3,
        [
            "previous_anchor_date",
            "date_0",
            "trading_gap"
        ]
    ]
    .reset_index(drop=True)
)

Unique TTF anchor dates: 130

Trading-observation gap from previous anchor:
trading_gap
1.0     41
2.0     19
3.0     21
4.0     12
5.0     10
6.0      6
7.0      4
8.0      2
9.0      4
10.0     1
11.0     1
12.0     1
13.0     1
15.0     1
18.0     1
20.0     1
21.0     1
23.0     1
30.0     1
Name: count, dtype: int64

Anchors immediately following another event anchor: 41
Anchors within two trading observations of previous anchor: 60
Anchors within three trading observations of previous anchor: 81

Closest consecutive event anchors:


,previous_anchor_date,date_0,trading_gap
0,2024-09-03,2024-09-04,1.0
1,2024-09-04,2024-09-09,3.0
2,2024-09-16,2024-09-19,3.0
3,2024-09-19,2024-09-23,2.0
4,2024-09-23,2024-09-24,1.0
...,...,...,...
76,2026-08-14,2026-08-16,1.0
77,2026-08-16,2026-08-19,3.0
78,2026-08-19,2026-08-20,1.0
79,2026-08-27,2026-08-28,1.0


### Overlapping-window findings

The 130 unique TTF anchor dates are frequently clustered in trading time. Of the 129 gaps between consecutive anchor dates, 60 are no more than two observed TTF trading intervals and 81 are no more than three.

This matters because the wider \([-1,+1]\) response window spans two consecutive observed-price return intervals. Event anchors separated by one or two trading observations therefore share part of the same market-return interval, creating mechanical dependence between those event windows.

A separation of three trading observations is the first point at which consecutive \([-1,+1]\) return intervals no longer overlap.

The full event sample is retained because clustered announcements remain economically relevant. However, dependence between overlapping market windows must be distinguished from the number of individual Gassco announcements. A non-overlapping event sample is therefore constructed as a robustness specification to remove mechanically shared return intervals between retained windows.

In [14]:
# Construct non-overlapping unique-anchor sample
# Requirement: retained anchors must be at least
# 3 TTF trading observations apart.

unique_anchors = (
    event_windows[
        [
            "date_0",
            "anchor_index",
            "date_m1",
            "date_p1",
            "date_p2",
            "price_m1",
            "price_0",
            "price_p1",
            "price_p2",
            "return_m1_0",
            "return_0_p1",
            "return_m1_p1",
            "return_0_p2"
        ]
    ]
    .drop_duplicates(subset=["anchor_index"])
    .sort_values("anchor_index")
    .reset_index(drop=True)
)

selected_rows = []
last_selected_index = None

for row in unique_anchors.itertuples():
    
    if (
        last_selected_index is None
        or row.anchor_index - last_selected_index >= 3
    ):
        selected_rows.append(row.Index)
        last_selected_index = row.anchor_index

nonoverlap_anchors = (
    unique_anchors
    .loc[selected_rows]
    .reset_index(drop=True)
)

# Validate spacing between retained anchors
nonoverlap_anchors["gap_from_previous_retained"] = (
    nonoverlap_anchors["anchor_index"].diff()
)

print("Unique event anchors:", len(unique_anchors))
print(
    "Non-overlapping retained anchors:",
    len(nonoverlap_anchors)
)

print(
    "Anchors excluded because of overlapping windows:",
    len(unique_anchors) - len(nonoverlap_anchors)
)

print("\nMinimum gap between retained anchors:")
print(
    nonoverlap_anchors[
        "gap_from_previous_retained"
    ].dropna().min()
)

print("\nRetained-anchor gap distribution:")
print(
    nonoverlap_anchors[
        "gap_from_previous_retained"
    ]
    .dropna()
    .value_counts()
    .sort_index()
)

display(
    nonoverlap_anchors[
        [
            "date_m1",
            "date_0",
            "date_p1",
            "anchor_index",
            "gap_from_previous_retained"
        ]
    ].head(15)
)

Unique event anchors: 130
Non-overlapping retained anchors: 82
Anchors excluded because of overlapping windows: 48

Minimum gap between retained anchors:
3.0

Retained-anchor gap distribution:
gap_from_previous_retained
3.0     21
4.0     14
5.0     15
6.0      9
7.0      5
8.0      2
9.0      2
10.0     3
11.0     1
12.0     1
14.0     2
15.0     1
20.0     2
21.0     1
24.0     1
31.0     1
Name: count, dtype: int64


,date_m1,date_0,date_p1,anchor_index,gap_from_previous_retained
0,2024-09-02,2024-09-03,2024-09-04,46,NaN
1,2024-09-06,2024-09-09,2024-09-10,50,4.0
2,2024-09-13,2024-09-16,2024-09-17,55,5.0
3,2024-09-18,2024-09-19,2024-09-20,58,3.0
4,2024-09-23,2024-09-24,2024-09-25,61,3.0
5,2024-09-27,2024-09-30,2024-10-01,65,4.0
6,2024-10-04,2024-10-07,2024-10-08,70,5.0
7,2024-10-11,2024-10-14,2024-10-15,75,5.0
8,2024-10-17,2024-10-18,2024-10-21,79,4.0
9,2024-10-25,2024-10-28,2024-10-29,85,6.0


## Final alignment design

The alignment procedure produces three distinct analytical levels.

The original event universe contains 166 eligible first-revision Gassco outage announcements. Because multiple announcements can map to the same TTF trading date, these correspond to 130 unique market anchor dates.

A non-overlapping robustness sample is also constructed for the wider \([-1,+1]\) response window. Retained anchors must be separated by at least three observed TTF trading intervals, ensuring that their \([-1,+1]\) windows do not share observed-price return intervals

Applying this rule retains 82 of the 130 unique market anchors and excludes 48. The minimum observed separation between retained anchors is three trading observations.

The three analytical levels therefore serve different purposes:

- **166 announcement-level observations:** outage characteristics, publication timing and event composition.
- **130 unique market anchors:** market-response analysis without repeatedly counting identical TTF windows created by same-anchor announcements.
- **82 non-overlapping market anchors:** robustness analysis designed to remove shared return intervals between retained \([-1,+1]\) windows.

The non-overlapping sample is treated as a robustness specification rather than a superior event definition. Its composition depends on a transparent chronological selection rule that retains the earliest eligible anchor when potential response windows conflict.

These alignment rules are applied consistently across the announcement-level, unique-anchor and non-overlapping samples.

In [15]:
# Save validated event-alignment outputs for downstream analysis

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------------
# 1. Announcement-level aligned data
# -----------------------------------

announcement_path = (
    PROCESSED_DIR / "gassco_ttf_event_windows_announcement_level.csv"
)

event_windows.to_csv(
    announcement_path,
    index=False
)


# -----------------------------------
# 2. Unique market-anchor data
# -----------------------------------

anchor_path = (
    PROCESSED_DIR / "gassco_ttf_event_windows_unique_anchors.csv"
)

unique_anchors.to_csv(
    anchor_path,
    index=False
)


# -----------------------------------
# 3. Non-overlapping robustness sample
# -----------------------------------

nonoverlap_path = (
    PROCESSED_DIR / "gassco_ttf_event_windows_nonoverlap.csv"
)

nonoverlap_anchors.to_csv(
    nonoverlap_path,
    index=False
)


# -----------------------------------
# 4. Confirm saved outputs
# -----------------------------------

print("Saved analytical datasets:")
print()

for path in [
    announcement_path,
    anchor_path,
    nonoverlap_path
]:
    print(path.name)
    print("  exists:", path.exists())
    print("  size:", path.stat().st_size, "bytes")

print("\nRows saved:")
print("Announcement level:", len(event_windows))
print("Unique anchors:", len(unique_anchors))
print("Non-overlapping anchors:", len(nonoverlap_anchors))

Saved analytical datasets:

gassco_ttf_event_windows_announcement_level.csv
  exists: True
  size: 90445 bytes
gassco_ttf_event_windows_unique_anchors.csv
  exists: True
  size: 20862 bytes
gassco_ttf_event_windows_nonoverlap.csv
  exists: True
  size: 13573 bytes

Rows saved:
Announcement level: 166
Unique anchors: 130
Non-overlapping anchors: 82
